In [3]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================
# LOAD ALL DATASETS
# ============================
customers = pd.read_csv('D:/Bharathi/python code/raw data/olist_customers_dataset.csv')
geolocation = pd.read_csv('D:/Bharathi/python code/raw data/olist_geolocation_dataset.csv')
order_items = pd.read_csv('D:/Bharathi/python code/raw data/olist_order_items_dataset.csv')
order_payments = pd.read_csv('D:/Bharathi/python code/raw data/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('D:/Bharathi/python code/raw data/olist_order_reviews_dataset.csv')
orders = pd.read_csv('D:/Bharathi/python code/raw data/olist_orders_dataset.csv')
sellers = pd.read_csv('D:/Bharathi/python code/raw data/olist_sellers_dataset.csv')
product_category = pd.read_csv('D:/Bharathi/python code/raw data/product_category_name_translation.csv')
products = pd.read_csv('D:/Bharathi/python code/raw data/olist_products_dataset.csv')

# Dictionary for looping
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": order_payments,
    "reviews": order_reviews,
    "geolocation": geolocation,
    "sellers": sellers,
    "product_category": product_category,
    "products": products
}

# ============================
# DATA UNDERSTANDING
# ============================

for name, df in datasets.items():
    print(f"\n==================== {name.upper()} ====================")

    # Shape
    print(f"Shape: {df.shape} (Rows: {df.shape[0]}, Columns: {df.shape[1]})")

    # Column names
    print("\nColumns:")
    print(df.columns.tolist())

    # Data types
    print("\nData Types:")
    print(df.dtypes)

    # Missing values
    print("\nMissing Values:")
    print(df.isnull().sum())

    # Missing percentage
    missing_pct = (df.isnull().sum() / len(df)) * 100
    print("\nMissing Percentage (%):")
    print(missing_pct.round(2))

    # Duplicates
    print(f"\nDuplicate Rows: {df.duplicated().sum()}")

    # Summary statistics
    print("\nSummary Statistics:")
    print(df.describe(include='all'))

    # Unique values per column
    print("\nUnique Values per Column:")
    for col in df.columns:
        print(f"{col}: {df[col].nunique()} unique values")

# ============================
# DATA BALANCE CHECK (for classification)
# Example: review_score
# ============================

if "review_score" in order_reviews.columns:
    print("\nReview Score Balance:")
    print(order_reviews["review_score"].value_counts())

geolocation = (
    geolocation.dropna(subset=["geolocation_city", "geolocation_state"])
    .groupby("geolocation_zip_code_prefix")
    .agg({
        "geolocation_lat": "mean",
        "geolocation_lng": "mean",
        "geolocation_city": "first",
        "geolocation_state": "first",
    })
    .reset_index()
)

print("Basic data cleaning and deduplication done!")    

# ============================
# HANDLE MISSING VALUES
# ============================

# MCAR → Fill review comments
order_reviews['review_comment_title'] = order_reviews['review_comment_title'].fillna('no comment')
order_reviews['review_comment_message'] = order_reviews['review_comment_message'].fillna('no comment')

# MAR → Product metadata
products['product_category_name'] = products['product_category_name'].fillna('unknown')
products['product_name_lenght'] = products['product_name_lenght'].fillna(0)
products['product_description_lenght'] = products['product_description_lenght'].fillna(0)
products['product_photos_qty'] = products['product_photos_qty'].fillna(0)

# Numerical median imputation
dim_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in dim_cols:
    products[col] = products[col].fillna(products[col].median())

# MNAR → Orders timestamps
orders['order_approved_at'] = orders['order_approved_at'].fillna(orders['order_purchase_timestamp'])
# Merge Portuguese category names with English translations
products = products.merge(
    product_category, on="product_category_name", how="left"
)
products["product_category_name_english"] = products[
    "product_category_name_english"
].fillna("unknown")

# Filter out any negative price or freight values
order_items = order_items[order_items["price"] >= 0]
order_items = order_items[order_items["freight_value"] >= 0]

print("Missing values imputed successfully!")

# ============================
# REMOVE DUPLICATES
# ============================

for name, df in datasets.items():
    dupes = df.duplicated().sum()
    if dupes > 0:
        df.drop_duplicates(inplace=True)
        print(f"Dropped {dupes} duplicates from {name}")

# ============================
#  VERIFY CLEANING
# ============================

for name, df in datasets.items():
    print(f"\n=== {name.upper()} CLEANING VERIFICATION ===")
    print("Missing values:")
    print(df.isnull().sum())
    print("Duplicates:", df.duplicated().sum())

print("=== BASIC INFO ===")
df.info()

print("\n=== NUMERICAL SUMMARY ===")
display(df.describe(include=["string", "category", "object"]))

print("\n=== CATEGORICAL SUMMARY ===")
display(df.describe(include=["string", "category"]))




==================== CUSTOMERS ====================
Shape: (99441, 5) (Rows: 99441, Columns: 5)

Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Data Types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Missing Values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Missing Percentage (%):
customer_id                 0.0
customer_unique_id          0.0
customer_zip_code_prefix    0.0
customer_city               0.0
customer_state              0.0
dtype: float64

Duplicate Rows: 0

Summary Statistics:
                             customer_id                customer_unique_id  \
count                              99441                             99441   
unique             

,product_id,product_category_name
count,32951,32951
unique,32951,74
top,1e9e8ef04dbcff4541ed26657ea517e5,cama_mesa_banho
freq,1,3029



=== CATEGORICAL SUMMARY ===


,product_id,product_category_name
count,32951,32951
unique,32951,74
top,1e9e8ef04dbcff4541ed26657ea517e5,cama_mesa_banho
freq,1,3029
